# Case 1: Full Load Banco de Dados (`fornecedores_legado`)
* **Nome:** Valtair Armendani da Silva
* **RA:** 10781071

### Carregando credenciais do Postgres e MinIO através da env

In [1]:
import io
import os
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import boto3
import psycopg2
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"

if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Arquivo .env não encontrado em {ENV_FILE}")

load_dotenv(ENV_FILE, override=False)

def env(nome: str) -> str:
    valor = os.environ.get(nome, "").strip()
    if not valor:
        raise ValueError(f"Variável {nome} não definida no .env")
    return valor

print("Variáveis carregadas com sucesso!")

Variáveis carregadas com sucesso!


### Conexão read-only e Query SQL

In [2]:
conn = psycopg2.connect(
    host=env("POSTGRES_HOST"),
    port=env("POSTGRES_PORT"),
    dbname=env("POSTGRES_DB"),
    user=env("POSTGRES_USER"),
    password=env("POSTGRES_PASSWORD"),
    options="-c default_transaction_read_only=on",
)

try:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM fornecedores_legado;")
        total_origem = cur.fetchone()[0]

        cur.execute("""
            SELECT
                id_fornecedor,
                razao_social,
                cnpj,
                status_credito,
                limite_credito
            FROM fornecedores_legado;
        """)
        registros = cur.fetchall()
finally:
    conn.close()

print(f"Total contado no banco (COUNT): {total_origem}")
print(f"Total de linhas extraídas: {len(registros)}")

Total contado no banco (COUNT): 50
Total de linhas extraídas: 50


### Criação da Tabela PyArrow com Schema

In [3]:
schema_bronze = pa.schema([
    pa.field("id_fornecedor", pa.string()),
    pa.field("razao_social", pa.string()),
    pa.field("cnpj", pa.string()),
    pa.field("status_credito", pa.string()),
    pa.field("limite_credito", pa.decimal128(12, 2)),
])

colunas_dados = {
    "id_fornecedor": [r[0] for r in registros],
    "razao_social": [r[1] for r in registros],
    "cnpj": [r[2] for r in registros],
    "status_credito": [r[3] for r in registros],
    "limite_credito": [Decimal(str(r[4])) if r[4] is not None else None for r in registros],
}

tabela_arrow = pa.Table.from_pydict(colunas_dados, schema=schema_bronze)
print(tabela_arrow.schema)

id_fornecedor: string
razao_social: string
cnpj: string
status_credito: string
limite_credito: decimal128(12, 2)


### Validação

In [4]:
assert tabela_arrow.num_rows == total_origem, (
    f"Divergência de volumetria! Origem: {total_origem}, Extraído: {tabela_arrow.num_rows}"
)

print(f"Validação concluída com sucesso: {tabela_arrow.num_rows} registros íntegros.")

Validação concluída com sucesso: 50 registros íntegros.


### Upload Parquet e _SUCCESS

In [5]:
s3 = boto3.client(
    "s3",
    endpoint_url=env("S3_ENDPOINT_URL"),
    aws_access_key_id=env("MINIO_ROOT_USER"),
    aws_secret_access_key=env("MINIO_ROOT_PASSWORD"),
)

bucket_bronze = env("MINIO_BUCKET_BRONZE")
agora_utc = datetime.now(timezone.utc)
run_id = agora_utc.strftime("%Y%m%dT%H%M%SZ")

prefixo = (
    f"fornecedores_legado/"
    f"ano={agora_utc:%Y}/"
    f"mes={agora_utc:%m}/"
    f"dia={agora_utc:%d}/"
    f"execucao={run_id}"
)

key_parquet = f"{prefixo}/fornecedores_legado.parquet"
key_success = f"{prefixo}/_SUCCESS"

# 1. Serializa o PyArrow Table em memória e grava o Parquet
buffer_parquet = io.BytesIO()
pq.write_table(tabela_arrow, buffer_parquet)
buffer_parquet.seek(0)

s3.put_object(
    Bucket=bucket_bronze,
    Key=key_parquet,
    Body=buffer_parquet.getvalue(),
)

# 2. Grava o _SUCCESS (0 bytes) somente após o Parquet estar salvo
s3.put_object(
    Bucket=bucket_bronze,
    Key=key_success,
    Body=b"",
)

print(f"Snapshot gravado com sucesso em: s3://{bucket_bronze}/{prefixo}/")

Snapshot gravado com sucesso em: s3://bronze/fornecedores_legado/ano=2026/mes=11/dia=17/execucao=20261117T004022Z/


### Leitura e Conferência do MinIO

In [6]:
# 1. Lista os objetos na pasta da execução
objetos = s3.list_objects_v2(Bucket=bucket_bronze, Prefix=f"{prefixo}/")
print("Arquivos gravados na pasta:")
for obj in objetos.get("Contents", []):
    print(f" - {obj['Key']} ({obj['Size']} bytes)")

# 2. Lê o Parquet de volta do MinIO
resposta = s3.get_object(Bucket=bucket_bronze, Key=key_parquet)
tabela_lida = pq.read_table(io.BytesIO(resposta["Body"].read()))

print("\n--- Conferência do Parquet ---")
print(f"Contagem de linhas: {tabela_lida.num_rows}")
print(f"\nSchema:\n{tabela_lida.schema}")
print("\n5 primeiras linhas:")
display(tabela_lida.to_pandas().head(5))

Arquivos gravados na pasta:
 - fornecedores_legado/ano=2026/mes=11/dia=17/execucao=20261117T004022Z/_SUCCESS (0 bytes)
 - fornecedores_legado/ano=2026/mes=11/dia=17/execucao=20261117T004022Z/fornecedores_legado.parquet (4151 bytes)

--- Conferência do Parquet ---
Contagem de linhas: 50

Schema:
id_fornecedor: string
razao_social: string
cnpj: string
status_credito: string
limite_credito: decimal128(12, 2)

5 primeiras linhas:


,id_fornecedor,razao_social,cnpj,status_credito,limite_credito
0,FORN-BDD640FB06,Alves e Filhos Ltda.,39250467000125,EM_ANALISE,174344.14
1,FORN-1A3D1FA7BC,Andrade S/A,13846092000177,BLOQUEADO,375989.28
2,FORN-0822E8F36C,Câmara - ME,94061325000128,APROVADO,72877.81
3,FORN-9A1DE64481,Cunha Teixeira Ltda. S/A,16928450000189,APROVADO,386645.18
4,FORN-B38A088CA6,Nascimento S/A,61495723000101,BLOQUEADO,291534.98
